# 0 - Mount

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# List files in a directory
!ls '/content/drive/My Drive/Project_SS24/MedAugment'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 classification.py  'My experiement.ipynb'   Running_code.ipynb        utils
 dataset	     recording		     Segmentation_Code.ipynb
 Inf.gdoc	     requirements.txt	     segmentation.py


# Check the GPU

In [ ]:
import torch
print("Is GPU available:", torch.cuda.is_available())

Is GPU available: True


In [ ]:
!pip install SimpleITK
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 MB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.1 MB/s eta 0:00:00
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16425 sha256=b793275f8b05f91de7e95c269aa1710b40c874d7aaef252a809fa035530ddcaf
  Stored in directory: /root/.cache/pip/wheels/03/3f/e9/911b1bc46869644912bda90a56bcf7b960f20b5187feea3baf
  Created wheel for pretrainedmodels: filename=pretrainedmodels-0.7.4-py3-none-any.whl size=60944 sha256=e69d80aceacd56d018db9542e29a35dd5db953fe91c619333d44b3b62f05dcbd
  Stored in directory: /root/.cache/pip/wheels/35/cb/a5/8f534c601428

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Project_SS24/MedAugment')


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Project_SS24/MedAugment/utils')


In [ ]:
!python medaugment.py --dataset LUNG --train_type segmentation

Executing data augmentation for image segmentation...
[ WARN:0@333.667] global loadsave.cpp:241 findDecoder imread_('/content/drive/MyDrive/Project_SS24/MedAugment/dataset/segmentation/LUNG/baseline/training_mask/MERS (88)_mask.png'): can't open/read file: check file path/integrity
Traceback (most recent call last):
  File "/content/drive/MyDrive/Project_SS24/MedAugment/utils/medaugment.py", line 144, in <module>
    main()
  File "/content/drive/MyDrive/Project_SS24/MedAugment/utils/medaugment.py", line 140, in main
    generate_datasets(**vars(args))
  File "/content/drive/MyDrive/Project_SS24/MedAugment/utils/medaugment.py", line 128, in generate_datasets
    med_augment(data_path, name, level, number_branch, mask_i=True)
  File "/content/drive/MyDrive/Project_SS24/MedAugment/utils/medaugment.py", line 79, in med_augment
    transformed = img_transform(image=image, mask=mask)
  File "/usr/local/lib/python3.10/dist-packages/albumentations/core/composition.py", line 304, in __call__
 

# Label-Aware Cropping & Resizing: (Segmentation)

# Zoom out (label aware segmentation)

In [ ]:
from PIL import Image
import numpy as np
import os

# Function to apply zoom-out effect
def zoom_out(image, target_size=(224, 224), zoomed_size=(150, 150)):
    """
    Resize the image to a smaller size and place it in the center of a new image with margins filled with the average color.
    Args:
        image (PIL Image): The input image to zoom out.
        target_size (tuple): The size of the output image (width, height).
        zoomed_size (tuple): The size of the zoomed-out image (width, height).
    Returns:
        PIL Image: Image with zoom-out effect.
    """
    # Resize the image to the zoomed-out size
    zoomed_image = image.resize(zoomed_size, Image.BILINEAR)

    # Calculate the average color of the original image
    average_color = np.array(image).mean(axis=(0, 1)).astype(np.uint8)
    avg_color_img = Image.new('RGB', target_size, tuple(average_color))

    # Calculate the position to paste the zoomed image on the average color background
    paste_x = (target_size[0] - zoomed_size[0]) // 2
    paste_y = (target_size[1] - zoomed_size[1]) // 2

    # Paste the zoomed image onto the center of the average color background
    avg_color_img.paste(zoomed_image, (paste_x, paste_y))

    return avg_color_img

# Function to process all images in a folder and save the output
def process_folder(input_folder, mask_folder, output_image_folder, output_mask_folder):
    """
    Process all images in the input folder, apply the zoom-out effect, and save them with a '_6' suffix in the output folder.
    Args:
        input_folder (str): Path to the folder containing the original images.
        mask_folder (str): Path to the folder containing the mask images.
        output_image_folder (str): Path to the folder where the zoomed-out images will be saved.
        output_mask_folder (str): Path to the folder where the zoomed-out masks will be saved.
    """
    # Create output folders if they don't exist
    if not os.path.exists(output_image_folder):
        os.makedirs(output_image_folder)

    if not os.path.exists(output_mask_folder):
        os.makedirs(output_mask_folder)

    for filename in os.listdir(input_folder):
        if filename.endswith('.jpg'):
            # Define paths for input image and mask
            image_path = os.path.join(input_folder, filename)
            mask_path = os.path.join(mask_folder, filename.replace('.jpg', '_mask.jpg'))

            # Define paths for output image and mask
            output_image_path = os.path.join(output_image_folder, filename.replace('.jpg', '_6.jpg'))

            # Modify this line to save the mask with '_6_mask.png'
            output_mask_path = os.path.join(output_mask_folder, filename.replace('.jpg', '_6_mask.jpg'))

            # Open the image and mask
            image = Image.open(image_path).convert('RGB')
            mask = Image.open(mask_path).convert('L')

            # Apply zoom-out effect to image and mask
            zoomed_out_image = zoom_out(image, target_size=(224, 224), zoomed_size=(150, 150))
            zoomed_out_mask = zoom_out(mask.convert('RGB'), target_size=(224, 224), zoomed_size=(150, 150)).convert('L')

            # Save the zoomed-out image and mask
            zoomed_out_image.save(output_image_path)
            zoomed_out_mask.save(output_mask_path)

            print(f"Processed and saved: {output_image_path} and {output_mask_path}")

# Define paths
input_folder = './dataset/segmentation/kvasir/baseline/training/'
mask_folder = './dataset/segmentation/kvasir/baseline/training_mask/'
output_image_folder = './dataset/segmentation/kvasir/medaugment/training/'
output_mask_folder = './dataset/segmentation/kvasir/medaugment/training_mask/'

# Process all images in the folder
process_folder(input_folder, mask_folder, output_image_folder, output_mask_folder)


## Zoom-in (segmentation): label-aware

In [ ]:
import cv2
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt

def label_aware_zoom(image, mask, padding=10, target_size=(224, 224)):
    # Find the bounding box of the important part (value 1 or 255) in the mask
    coords = np.column_stack(np.where((mask == 1) | (mask == 255)))

    # If there are no '1' or '255' pixels in the mask, return the original image and mask
    if coords.size == 0:
        print("Warning: No important regions found in the mask.")
        return image, mask

    top_left = coords.min(axis=0)
    bottom_right = coords.max(axis=0)

    # Optionally, expand the bounding box by a padding value (be careful with image boundaries)
    top_left = np.maximum(top_left - padding, 0)
    bottom_right = np.minimum(bottom_right + padding, mask.shape)

    # Crop the mask and image using the bounding box
    cropped_mask = mask[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1]]
    cropped_image = image[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1]]

    # Resize the cropped image and mask to the target size (224x224)
    resized_image = cv2.resize(cropped_image, target_size, interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(cropped_mask, target_size, interpolation=cv2.INTER_NEAREST)

    return resized_image, resized_mask

def process_folder(image_folder, mask_folder, output_image_folder, output_mask_folder, padding=10, target_size=(224, 224)):
    # Ensure output folders exist
    os.makedirs(output_image_folder, exist_ok=True)
    os.makedirs(output_mask_folder, exist_ok=True)

    # Iterate over all files in the image folder
    for image_file in os.listdir(image_folder):
        if image_file.endswith(('.png', '.jpg', '.jpeg')):  # Filter image files
            # Load corresponding mask file
            mask_file = image_file.replace('.jpg', '_mask.jpg')  # Adjust this if needed
            image_path = os.path.join(image_folder, image_file)
            mask_path = os.path.join(mask_folder, mask_file)

            if not os.path.exists(mask_path):
                print(f"Mask for {image_file} not found. Skipping.")
                continue

            # Load the image and mask in their original format
            image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)  # Load the image in its original color (RGB or grayscale)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)    # Load the mask in grayscale

            # Apply zoom-in and resizing
            resized_image, resized_mask = label_aware_zoom(image, mask, padding=padding, target_size=target_size)

            # Define new filenames with the desired suffixes
            image_filename_7 = os.path.splitext(image_file)[0] + '_7.jpg'  # Add _7 suffix
            mask_filename_7 = os.path.splitext(image_file)[0] + '_7_mask.jpg'  # Add _7_mask suffix

            # Save the processed image and mask
            output_image_path = os.path.join(output_image_folder, image_filename_7)
            output_mask_path = os.path.join(output_mask_folder, mask_filename_7)

            cv2.imwrite(output_image_path, resized_image)
            cv2.imwrite(output_mask_path, resized_mask)

            print(f"Processed and saved: {image_filename_7} and {mask_filename_7}")

# Example usage:
image_folder = './dataset/segmentation/kvasir/baseline/training/'  # Input folder for images
mask_folder = './dataset/segmentation/kvasir/baseline/training_mask/'    # Input folder for masks
output_image_folder = './dataset/segmentation/kvasir/medaugment/training/'  # Output folder for processed images
output_mask_folder = './dataset/segmentation/kvasir/medaugment/training_mask/'    # Output folder for processed masks

# Process the entire folder
process_folder(image_folder, mask_folder, output_image_folder, output_mask_folder)


Processed and saved: cju5hyi9yegob0755ho3do8en_7.jpg and cju5hyi9yegob0755ho3do8en_7_mask.jpg
Processed and saved: cju5hqz50e7o90850e0prlpa0_7.jpg and cju5hqz50e7o90850e0prlpa0_7_mask.jpg
Processed and saved: cju5hwonqedw10801vsd3w6kk_7.jpg and cju5hwonqedw10801vsd3w6kk_7_mask.jpg
Processed and saved: cju5i5oh2efg60987ez6cpf72_7.jpg and cju5i5oh2efg60987ez6cpf72_7_mask.jpg
Processed and saved: cju5i39mreass0817au8p22zy_7.jpg and cju5i39mreass0817au8p22zy_7_mask.jpg
Processed and saved: cju5hi52odyf90817prvcwg45_7.jpg and cju5hi52odyf90817prvcwg45_7_mask.jpg
Processed and saved: cju5ht88gedbu0755xrcuddcx_7.jpg and cju5ht88gedbu0755xrcuddcx_7_mask.jpg
Processed and saved: cju5huurrecm70801y680y13m_7.jpg and cju5huurrecm70801y680y13m_7_mask.jpg
Processed and saved: cju5hjxaae3i40850h5z2laf5_7.jpg and cju5hjxaae3i40850h5z2laf5_7_mask.jpg
Processed and saved: cju5jx7jzf7c90871c2i9aiov_7.jpg and cju5jx7jzf7c90871c2i9aiov_7_mask.jpg
Processed and saved: cju5hl8nee8a40755fm8qjj0o_7.jpg and cju

Resizing

In [ ]:
from PIL import Image
import os

# Define the path to the folder containing the images
folder_path = './dataset/segmentation/kvasir/medaugment/training_mask/'  # Replace with your actual folder path

# Target size for resizing
target_size = (224, 224)

# Get a list of all files in the folder
files_in_folder = os.listdir(folder_path)

# Loop through all files in the folder
for filename in files_in_folder:
    # Check if the file is an image (you can add more extensions if needed)
    if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')):
        # Construct the full path to the image file
        file_path = os.path.join(folder_path, filename)

        # Open the image using PIL
        with Image.open(file_path) as img:
            # Get the size of the image
            width, height = img.size

            # Print the filename and its size
            print(f"Image: {filename}, Size: {width}x{height}")

            # Check if the image size is not 224x224
            if (width, height) != target_size:
                print(f"Resizing image: {filename}")

                # Resize the image to 224x224
                resized_img = img.resize(target_size, Image.BILINEAR)

                # Save the resized image back to the same path (overwrite the original)
                resized_img.save(file_path)
                print(f"Image resized and saved: {filename}")

print("Image size check and resizing process complete.")


Streaming output truncated to the last 5000 lines.
Image resized and saved: cju2qdj95ru8g09886gfi9rsz_3_mask.jpg
Image: cju2qdj95ru8g09886gfi9rsz_4_mask.jpg, Size: 571x531
Resizing image: cju2qdj95ru8g09886gfi9rsz_4_mask.jpg
Image resized and saved: cju2qdj95ru8g09886gfi9rsz_4_mask.jpg
Image: cju2pkwt3r8b90988v2ywq1px_1_mask.jpg, Size: 622x530
Resizing image: cju2pkwt3r8b90988v2ywq1px_1_mask.jpg
Image resized and saved: cju2pkwt3r8b90988v2ywq1px_1_mask.jpg
Image: cju2pkwt3r8b90988v2ywq1px_5_mask.jpg, Size: 622x530
Resizing image: cju2pkwt3r8b90988v2ywq1px_5_mask.jpg
Image resized and saved: cju2pkwt3r8b90988v2ywq1px_5_mask.jpg
Image: cju2pkwt3r8b90988v2ywq1px_2_mask.jpg, Size: 622x530
Resizing image: cju2pkwt3r8b90988v2ywq1px_2_mask.jpg
Image resized and saved: cju2pkwt3r8b90988v2ywq1px_2_mask.jpg
Image: cju2pkwt3r8b90988v2ywq1px_3_mask.jpg, Size: 622x530
Resizing image: cju2pkwt3r8b90988v2ywq1px_3_mask.jpg
Image resized and saved: cju2pkwt3r8b90988v2ywq1px_3_mask.jpg
Image: cju2pkwt3r

Check images and masks alignment

In [ ]:
import os

# Define the paths to the image and mask folders
image_folder = './dataset/segmentation/kvasir/medaugment/training/'  # Replace with your image folder path
mask_folder = './dataset/segmentation/kvasir/medaugment/training_mask/'  # Replace with your mask folder path

# Get a list of all images and masks
image_files = [f for f in os.listdir(image_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
mask_files = [f for f in os.listdir(mask_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

# Extract mask base names to check for their existence
mask_files_set = set(mask_files)  # Convert to a set for faster lookups


# List to store images without corresponding masks
images_without_masks = []

# Check for each image if the corresponding mask exists
for image_file in image_files:
    image_base_name = os.path.splitext(image_file)[0]  # Get the base name (e.g., "1" from "1.png")
    expected_mask_name = f"{image_base_name}_mask.jpg"  # Construct the expected mask filename (e.g., "1_mask.png")

    if expected_mask_name not in mask_files_set:
        images_without_masks.append(image_file)

# Output the results
Num = 0
if images_without_masks:
    print("Images without corresponding masks:")
    for image in images_without_masks:
        print(image)
        Num+=1
else:
    print("All images have corresponding masks.")
print(Num)

All images have corresponding masks.
0


Delete individual

In [ ]:
import os

# Define the path to the folder containing the images
folder_path = './dataset/segmentation/kvasir/medaugment/training_mask/'  # Replace with your actual folder path

# Define the base name to delete (all files starting with this name)
base_name_to_delete = 'cju5k3j3uf6de0817hszzfr7n_6_mask.jpg'

# Get a list of all files in the folder
files_in_folder = os.listdir(folder_path)

# Loop through all files and delete those starting with the base name
for filename in files_in_folder:
    if filename.startswith(base_name_to_delete):
        file_path = os.path.join(folder_path, filename)
        os.remove(file_path)
        print(f"Deleted file: {filename}")

print("Deletion process complete.")


Deleted file: cju5k3j3uf6de0817hszzfr7n_6_mask.jpg
Deletion process complete.


Delete images

In [ ]:
import os

# Define source directories for images and masks
folders_to_clean = [
    './dataset/segmentation/kvasir/medaugment/training/',  # Replace with your folder path for images
    './dataset/segmentation/kvasir/medaugment/training_mask/'  # Replace with your folder path for masks
]

# Define the suffixes to look for
suffixes_to_delete = ['_7', '_7_mask']

def delete_images_with_suffix(folder, suffixes):
    """
    Delete images in the specified folder with given suffixes.
    Args:
        folder (str): The folder path to search for images.
        suffixes (list of str): List of suffixes to delete.
    """
    for filename in os.listdir(folder):
        # Check if the file has one of the suffixes before the extension
        if any(filename.endswith(suffix + '.png') for suffix in suffixes):
            file_path = os.path.join(folder, filename)
            os.remove(file_path)
            print(f"Deleted: {file_path}")

# Delete images with specified suffixes in each folder
for folder in folders_to_clean:
    delete_images_with_suffix(folder, suffixes_to_delete)

print("Deletion complete.")


Deleted: ./dataset/segmentation/cvc/medaugment/training/105_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/104_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/103_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/1_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/102_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/10_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/101_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/100_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/126_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/127_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/125_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/124_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/121_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/118_7.png
Deleted: ./dataset/segmentation/cvc/medaugment/training/119_7.png
Deleted: ./da

In [ ]:
!python segmentation.py --model DeepLab --dataset LUNG --train_type segmentation --index 4 --bs 32 --num_class 1 --value 0.01 --epoch 40 --lr 0.002 --min_loss 10000

/usr/local/lib/python3.10/dist-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
76
Epoch: 0
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10
Epoch: 11
Epoch: 12
Epoch: 13
Epoch: 14
Epoch: 15
Epoch: 16
Epoch: 17
Epoch: 18
Epoch: 19
Epoch: 20
Epoch: 21
Epoch: 22
Epoch: 23
Epoch: 24
Epoch: 25
Epoch: 26
Epoch: 27
Epoch: 28


# Random Cropping & Resizing: (classification)

*   Zoom out effect (_6)
*   Zoom in effect (_7)

In [ ]:
from PIL import Image
import numpy as np
import os

# Define source and destination directories
source_folder = './dataset/classification/lung/baseline/training/n0/'  # Replace with your source folder path
destination_folder = './dataset/classification/lung/medaugment/training/n0/'  # Replace with your destination folder path

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

def zoom_out(image, target_size=(224, 224), zoomed_size=(150, 150)):
    """
    Resize the image to a smaller size and place it in the center of a new image with margins filled with the average color.
    Args:
        image (PIL Image): The input image to zoom out.
        target_size (tuple): The size of the output image (width, height).
        zoomed_size (tuple): The size of the zoomed-out image (width, height).
    Returns:
        PIL Image: Image with zoom-out effect.
    """
    # Resize the image to the zoomed-out size
    zoomed_image = image.resize(zoomed_size, Image.BILINEAR)

    # Calculate the average color of the original image
    average_color = np.array(image).mean(axis=(0, 1)).astype(np.uint8)
    avg_color_img = Image.new('RGB', target_size, tuple(average_color))

    # Calculate the position to paste the zoomed image on the average color background
    paste_x = (target_size[0] - zoomed_size[0]) // 2
    paste_y = (target_size[1] - zoomed_size[1]) // 2

    # Paste the zoomed image onto the center of the average color background
    avg_color_img.paste(zoomed_image, (paste_x, paste_y))

    return avg_color_img

def random_cropping(image, crop_size=(150, 150)):
    """
    Perform a random crop of the image to the specified size.
    Args:
        image (PIL Image): The input image to crop.
        crop_size (tuple): The size of the crop (width, height).
    Returns:
        PIL Image: Cropped image.
    """
    width, height = image.size

    # Calculate random top-left corner for cropping
    left = np.random.randint(0, width - crop_size[0] + 1)
    top = np.random.randint(0, height - crop_size[1] + 1)

    # Calculate the right and bottom points
    right = left + crop_size[0]
    bottom = top + crop_size[1]

    return image.crop((left, top, right, bottom))

def resize_image(image, target_size=(224, 224)):
    """
    Resize the image to the target size.
    Args:
        image (PIL Image): The input image to resize.
        target_size (tuple): The target size (width, height).
    Returns:
        PIL Image: Resized image.
    """
    return image.resize(target_size, Image.BILINEAR)

# Process each image in the source folder
for filename in os.listdir(source_folder):
    if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.jpeg'):
        image_path = os.path.join(source_folder, filename)

        # Load the image
        image = Image.open(image_path).convert('RGB')

        # Step 1: Resize image to 224x224 to ensure it is large enough for cropping
        resized_image = resize_image(image, target_size=(224, 224))

        # Step 2: Apply zoom-out effect
        zoomed_out_image = zoom_out(resized_image, target_size=(224, 224), zoomed_size=(150, 150))

        # Save the zoomed-out image to the destination folder with "_6" suffix
        base_name = os.path.splitext(filename)[0]
        zoomed_out_image.save(os.path.join(destination_folder, f"{base_name}_6.png"))
        print(f"Processed and saved zoom-out: {base_name}_6.png")

        # Step 3: Apply random cropping (zoom-in effect)
        cropped_image = random_cropping(resized_image, crop_size=(150, 150))

        # Step 4: Resize cropped image back to 224x224
        final_zoom_in_image = resize_image(cropped_image, target_size=(224, 224))

        # Save the zoomed-in image to the destination folder with "_7" suffix
        final_zoom_in_image.save(os.path.join(destination_folder, f"{base_name}_7.png"))
        print(f"Processed and saved zoom-in: {base_name}_7.png")

print("Processing complete.")


Processed and saved zoom-out: COVID (174)_6.png
Processed and saved zoom-in: COVID (174)_7.png
Processed and saved zoom-out: COVID (172)_6.png
Processed and saved zoom-in: COVID (172)_7.png
Processed and saved zoom-out: COVID (178)_6.png
Processed and saved zoom-in: COVID (178)_7.png
Processed and saved zoom-out: COVID (171)_6.png
Processed and saved zoom-in: COVID (171)_7.png
Processed and saved zoom-out: COVID (175)_6.png
Processed and saved zoom-in: COVID (175)_7.png
Processed and saved zoom-out: COVID (177)_6.png
Processed and saved zoom-in: COVID (177)_7.png
Processed and saved zoom-out: COVID (179)_6.png
Processed and saved zoom-in: COVID (179)_7.png
Processed and saved zoom-out: COVID (176)_6.png
Processed and saved zoom-in: COVID (176)_7.png
Processed and saved zoom-out: COVID (169)_6.png
Processed and saved zoom-in: COVID (169)_7.png
Processed and saved zoom-out: COVID (173)_6.png
Processed and saved zoom-in: COVID (173)_7.png
Processed and saved zoom-out: COVID (170)_6.png
Pr

In [ ]:
!python medaugment.py --dataset lung --train_type classification

Executing data augmentation for image classification...


In [ ]:
import os
from PIL import Image
import cv2

image_folder = '/content/drive/My Drive/Project_SS24/MedAugment/dataset/classification/lung/medaugment/training/n0'

# Create a new folder for the resized images if it doesn't exist
resized_folder = '/content/drive/My Drive/Project_SS24/MedAugment/dataset/classification/lung/medaugment/training/n0'
if not os.path.exists(resized_folder):
    os.makedirs(resized_folder)

# Iterate over each file in the folder
for filename in os.listdir(image_folder):
    if filename.endswith(".jpg") or filename.endswith(".png"):  # You can add more image extensions if needed
        image_path = os.path.join(image_folder, filename)
        img = Image.open(image_path)

        # Resize the image to 224x224
        img_resized = img.resize((224, 224))

        # Save the resized image to the new folder
        img_resized.save(os.path.join(resized_folder, filename))


In [ ]:
from PIL import Image
import os

# Define the directory where the images are stored
image_directory = '/content/drive/My Drive/Project_SS24/MedAugment/dataset/classification/cataract/medaugment/test/n0'

# Loop through all files in the directory
for filename in os.listdir(image_directory):
    if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):  # Filter only image files
        file_path = os.path.join(image_directory, filename)

        # Open an image file
        with Image.open(file_path) as img:
            # Get image size
            width, height = img.size
            print(f'Image: {filename} | Size: {width}x{height} pixels')


Image: image_251.png | Size: 597x313 pixels
Image: image_247.png | Size: 1920x1080 pixels
Image: image_248.png | Size: 999x650 pixels
Image: image_252.png | Size: 1600x1067 pixels
Image: image_253.png | Size: 450x450 pixels
Image: image_249.png | Size: 321x307 pixels
Image: image_250.png | Size: 1125x750 pixels
Image: image_246.png | Size: 1200x800 pixels
Image: image_257.png | Size: 250x175 pixels
Image: image_263.png | Size: 800x530 pixels
Image: image_267.png | Size: 900x600 pixels
Image: image_258.png | Size: 250x175 pixels
Image: image_265.png | Size: 800x539 pixels
Image: image_264.png | Size: 241x183 pixels
Image: image_255.png | Size: 400x260 pixels
Image: image_259.png | Size: 615x409 pixels
Image: image_268.png | Size: 1200x800 pixels
Image: image_261.png | Size: 800x533 pixels
Image: image_260.png | Size: 630x430 pixels
Image: image_266.png | Size: 615x409 pixels
Image: image_262.png | Size: 557x415 pixels
Image: image_256.png | Size: 1000x667 pixels
Image: image_270.png | S

In [ ]:
!python classification.py --model VGGNet --dataset btmri --train_type classification --index 5 --seed 1 --decay True --bs 128 --num_class 4 --value 0.01 --epoch 40 --lr 0.002 --min_loss 10000

/usr/local/lib/python3.10/dist-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
Epoch: 0
Step 0, Training loss: 1.392
Step 10, Training loss: 0.936
Step 20, Training loss: 0.661
Step 30, Training loss: 0.486
Step 40, Training loss: 0.476
Step 50, Training loss: 0.539
Step 60, Training loss: 0.354
Step 70, Training loss: 0.417
Step 80, Training loss: 0.466
Step 90, Training loss: 0.346
Step 100, Training loss: 0.443
Step 110, Training loss: 0.496
Step 120, Training loss: 0.413
Step 130, Training loss: 0.407
Step 140, Training loss: 0.356
Step 150, Training loss: 0.466
Step 160, Training loss: 0.454
Step 170, Training loss: 0.316
Step 180, Training loss: 0.374
Step 190, Training loss: 0.421
Step 200, Training loss: 0.387
Step 210, Training los

# Random Cropping & Resizing: (Segmentation)

In [ ]:
from PIL import Image
import numpy as np
import os

# Define source and destination directories for images and masks
source_folder_images = './dataset/segmentation/kvasir/baseline/training/'  # Replace with your source folder path for images
source_folder_masks = './dataset/segmentation/kvasir/baseline/training_mask/'  # Replace with your source folder path for masks

destination_folder_images = './dataset/segmentation/kvasir/medaugment/training/'  # Replace with your destination folder path for images
destination_folder_masks = './dataset/segmentation/kvasir/medaugment/training_mask/'  # Replace with your destination folder path for masks

# Ensure the destination folders exist
os.makedirs(destination_folder_images, exist_ok=True)
os.makedirs(destination_folder_masks, exist_ok=True)

def zoom_out(image, target_size=(224, 224), zoomed_size=(150, 150)):
    """
    Resize the image to a smaller size and place it in the center of a new image with margins filled with the average color.
    Args:
        image (PIL Image): The input image to zoom out.
        target_size (tuple): The size of the output image (width, height).
        zoomed_size (tuple): The size of the zoomed-out image (width, height).
    Returns:
        PIL Image: Image with zoom-out effect.
    """
    zoomed_image = image.resize(zoomed_size, Image.BILINEAR)
    average_color = np.array(image).mean(axis=(0, 1)).astype(np.uint8)
    avg_color_img = Image.new('RGB', target_size, tuple(average_color))
    paste_x = (target_size[0] - zoomed_size[0]) // 2
    paste_y = (target_size[1] - zoomed_size[1]) // 2
    avg_color_img.paste(zoomed_image, (paste_x, paste_y))
    return avg_color_img

def random_cropping(image, crop_size=(150, 150)):
    """
    Perform a random crop of the image to the specified size.
    Args:
        image (PIL Image): The input image to crop.
        crop_size (tuple): The size of the crop (width, height).
    Returns:
        PIL Image: Cropped image.
    """
    width, height = image.size
    left = np.random.randint(0, width - crop_size[0] + 1)
    top = np.random.randint(0, height - crop_size[1] + 1)
    right = left + crop_size[0]
    bottom = top + crop_size[1]
    return image.crop((left, top, right, bottom))

def resize_image(image, target_size=(224, 224)):
    """
    Resize the image to the target size.
    Args:
        image (PIL Image): The input image to resize.
        target_size (tuple): The target size (width, height).
    Returns:
        PIL Image: Resized image.
    """
    return image.resize(target_size, Image.BILINEAR)

# Function to process images or masks
def process_and_save(image, base_name, destination_folder, suffix):
    """
    Process the image or mask and save it with a given suffix.
    Args:
        image (PIL Image): The input image to process.
        base_name (str): The base name of the file.
        destination_folder (str): The folder to save processed images.
        suffix (str): The suffix to add to the filename.
    """
    resized_image = resize_image(image, target_size=(224, 224))
    if suffix == '_6':
        processed_image = zoom_out(resized_image, target_size=(224, 224), zoomed_size=(150, 150))
    elif suffix == '_7':
        cropped_image = random_cropping(resized_image, crop_size=(150, 150))
        processed_image = resize_image(cropped_image, target_size=(224, 224))

    # Adjust filename to put the suffix before '_mask'
    if '_mask' in base_name:
        new_base_name = base_name.replace('_mask', f"{suffix}_mask")
    else:
        new_base_name = base_name + suffix

    processed_image.save(os.path.join(destination_folder, f"{new_base_name}.jpg"))
    print(f"Processed and saved: {new_base_name}.jpg")

# Process each image in the source folder for images and masks
for filename in os.listdir(source_folder_images):
    if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.jpeg'):
        base_name = os.path.splitext(filename)[0]

        # Process image
        image_path = os.path.join(source_folder_images, filename)
        image = Image.open(image_path).convert('RGB')
        process_and_save(image, base_name, destination_folder_images, '_6')
        process_and_save(image, base_name, destination_folder_images, '_7')

# Process each mask in the source folder
for filename in os.listdir(source_folder_masks):
    if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.jpeg'):
        base_name = os.path.splitext(filename)[0]

        # Process mask (assuming masks have the same filename in the mask folder)
        mask_path = os.path.join(source_folder_masks, filename)
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert('RGB')
            process_and_save(mask, base_name, destination_folder_masks, '_6')
            process_and_save(mask, base_name, destination_folder_masks, '_7')

print("Processing complete.")


Processed and saved: cju5hyi9yegob0755ho3do8en_6.jpg
Processed and saved: cju5hyi9yegob0755ho3do8en_7.jpg
Processed and saved: cju5hqz50e7o90850e0prlpa0_6.jpg
Processed and saved: cju5hqz50e7o90850e0prlpa0_7.jpg
Processed and saved: cju5hwonqedw10801vsd3w6kk_6.jpg
Processed and saved: cju5hwonqedw10801vsd3w6kk_7.jpg
Processed and saved: cju5i5oh2efg60987ez6cpf72_6.jpg
Processed and saved: cju5i5oh2efg60987ez6cpf72_7.jpg
Processed and saved: cju5i39mreass0817au8p22zy_6.jpg
Processed and saved: cju5i39mreass0817au8p22zy_7.jpg
Processed and saved: cju5hi52odyf90817prvcwg45_6.jpg
Processed and saved: cju5hi52odyf90817prvcwg45_7.jpg
Processed and saved: cju5ht88gedbu0755xrcuddcx_6.jpg
Processed and saved: cju5ht88gedbu0755xrcuddcx_7.jpg
Processed and saved: cju5huurrecm70801y680y13m_6.jpg
Processed and saved: cju5huurrecm70801y680y13m_7.jpg
Processed and saved: cju5hjxaae3i40850h5z2laf5_6.jpg
Processed and saved: cju5hjxaae3i40850h5z2laf5_7.jpg
Processed and saved: cju5jx7jzf7c90871c2i9aiov

In [ ]:
import os

# Define the path to the folder containing the images
folder_path = './dataset/segmentation/LUNG/medaugment/validation_mask/'  # Replace with your actual folder path

# Get a list of all files in the folder
files_in_folder = os.listdir(folder_path)

# Loop through all files in the folder
for filename in files_in_folder:
    # Check if the filename has the extra '_mask' suffix
    if filename.endswith('_mask_mask.png'):
        # Construct the new filename by removing the extra '_mask' part
        new_filename = filename.replace('_mask_mask.png', '_mask.png')

        # Define the full path for the current and new filenames
        old_file_path = os.path.join(folder_path, filename)
        new_file_path = os.path.join(folder_path, new_filename)

        # Rename the file
        os.rename(old_file_path, new_file_path)
        print(f"Renamed file: {filename} to {new_filename}")

print("Renaming process complete.")


Renamed file: COVID (281)_mask_mask.png to COVID (281)_mask.png
Renamed file: COVID (282)_mask_mask.png to COVID (282)_mask.png
Renamed file: COVID (280)_mask_mask.png to COVID (280)_mask.png
Renamed file: COVID (276)_mask_mask.png to COVID (276)_mask.png
Renamed file: COVID (278)_mask_mask.png to COVID (278)_mask.png
Renamed file: COVID (277)_mask_mask.png to COVID (277)_mask.png
Renamed file: COVID (274)_mask_mask.png to COVID (274)_mask.png
Renamed file: COVID (275)_mask_mask.png to COVID (275)_mask.png
Renamed file: COVID (279)_mask_mask.png to COVID (279)_mask.png
Renamed file: COVID (273)_mask_mask.png to COVID (273)_mask.png
Renamed file: COVID (272)_mask_mask.png to COVID (272)_mask.png
Renamed file: COVID (264)_mask_mask.png to COVID (264)_mask.png
Renamed file: COVID (267)_mask_mask.png to COVID (267)_mask.png
Renamed file: COVID (268)_mask_mask.png to COVID (268)_mask.png
Renamed file: COVID (269)_mask_mask.png to COVID (269)_mask.png
Renamed file: COVID (266)_mask_mask.png 

In [ ]:
import os

# Define the paths to the image and mask folders
image_folder = './dataset/segmentation/kvasir/medaugment/validation/'  # Replace with your image folder path
mask_folder = './dataset/segmentation/kvasir/medaugment/validation_mask/'  # Replace with your mask folder path

# Get a list of all images and masks
image_files = [f for f in os.listdir(image_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
mask_files = [f for f in os.listdir(mask_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

# Extract mask base names to check for their existence
mask_files_set = set(mask_files)  # Convert to a set for faster lookups


# List to store images without corresponding masks
images_without_masks = []

# Check for each image if the corresponding mask exists
for image_file in image_files:
    image_base_name = os.path.splitext(image_file)[0]  # Get the base name (e.g., "1" from "1.png")
    expected_mask_name = f"{image_base_name}_mask.jpg"  # Construct the expected mask filename (e.g., "1_mask.png")

    if expected_mask_name not in mask_files_set:
        images_without_masks.append(image_file)

# Output the results
Num = 0
if images_without_masks:
    print("Images without corresponding masks:")
    for image in images_without_masks:
        print(image)
        Num+=1
else:
    print("All images have corresponding masks.")
print(Num)

All images have corresponding masks.
0


In [ ]:
import os

# Define the path to the folder you want to check
folder_path = './dataset/segmentation/LUNG/medaugment/training/'  # Replace with your actual folder path

# Define the filenames to check for
required_images = ['MERS (88)_6.png', 'MERS (88)_7.png']

# Get a list of all files in the folder
files_in_folder = os.listdir(folder_path)

# Check if both required images are in the folder
missing_images = [image for image in required_images if image not in files_in_folder]

if not missing_images:
    print("Both images 'MERS (88)_6.png' and 'MERS (88)_7.png' are present in the folder.")
else:
    print("The following images are missing from the folder:")
    for missing_image in missing_images:
        print(missing_image)


The following images are missing from the folder:
MERS (88)_6.png
MERS (88)_7.png


In [ ]:
import os

# Define the path to the folder containing the images
folder_path = './dataset/segmentation/LUNG/medaugment/training/'  # Replace with your actual folder path

# Define the base name to delete (all files starting with this name)
base_name_to_delete = 'MERS (88)'

# Get a list of all files in the folder
files_in_folder = os.listdir(folder_path)

# Loop through all files and delete those starting with the base name
for filename in files_in_folder:
    if filename.startswith(base_name_to_delete):
        file_path = os.path.join(folder_path, filename)
        os.remove(file_path)
        print(f"Deleted file: {filename}")

print("Deletion process complete.")


Deleted file: MERS (88)_6.png
Deleted file: MERS (88)_7.png
Deletion process complete.


In [ ]:
from PIL import Image
import numpy as np
import os

# Define source and destination directories for images and masks
source_folder_images = './dataset/segmentation/LUNG/baseline/training/'  # Replace with your source folder path for images
source_folder_masks = './dataset/segmentation/LUNG/baseline/training_mask/'  # Replace with your source folder path for masks

destination_folder_images = './dataset/segmentation/LUNG/medaugment/training/'  # Replace with your destination folder path for images
destination_folder_masks = './dataset/segmentation/LUNG/medaugment/training_mask/'  # Replace with your destination folder path for masks

# Ensure the destination folders exist
os.makedirs(destination_folder_images, exist_ok=True)
os.makedirs(destination_folder_masks, exist_ok=True)

def zoom_out(image, target_size=(224, 224), zoomed_size=(150, 150)):
    """
    Resize the image to a smaller size and place it in the center of a new image with margins filled with the average color.
    Args:
        image (PIL Image): The input image to zoom out.
        target_size (tuple): The size of the output image (width, height).
        zoomed_size (tuple): The size of the zoomed-out image (width, height).
    Returns:
        PIL Image: Image with zoom-out effect.
    """
    zoomed_image = image.resize(zoomed_size, Image.BILINEAR)
    average_color = np.array(image).mean(axis=(0, 1)).astype(np.uint8)
    avg_color_img = Image.new('RGB', target_size, tuple(average_color))
    paste_x = (target_size[0] - zoomed_size[0]) // 2
    paste_y = (target_size[1] - zoomed_size[1]) // 2
    avg_color_img.paste(zoomed_image, (paste_x, paste_y))
    return avg_color_img

def random_cropping(image, crop_size=(150, 150)):
    """
    Perform a random crop of the image to the specified size.
    Args:
        image (PIL Image): The input image to crop.
        crop_size (tuple): The size of the crop (width, height).
    Returns:
        PIL Image: Cropped image.
    """
    width, height = image.size
    left = np.random.randint(0, width - crop_size[0] + 1)
    top = np.random.randint(0, height - crop_size[1] + 1)
    right = left + crop_size[0]
    bottom = top + crop_size[1]
    return image.crop((left, top, right, bottom))

def resize_image(image, target_size=(224, 224)):
    """
    Resize the image to the target size.
    Args:
        image (PIL Image): The input image to resize.
        target_size (tuple): The target size (width, height).
    Returns:
        PIL Image: Resized image.
    """
    return image.resize(target_size, Image.BILINEAR)

# Function to process images or masks
def process_and_save(image, base_name, destination_folder, suffix):
    """
    Process the image or mask and save it with a given suffix.
    Args:
        image (PIL Image): The input image to process.
        base_name (str): The base name of the file.
        destination_folder (str): The folder to save processed images.
        suffix (str): The suffix to add to the filename.
    """
    resized_image = resize_image(image, target_size=(224, 224))
    if suffix == '_6':
        processed_image = zoom_out(resized_image, target_size=(224, 224), zoomed_size=(150, 150))
    elif suffix == '_7':
        cropped_image = random_cropping(resized_image, crop_size=(150, 150))
        processed_image = resize_image(cropped_image, target_size=(224, 224))

    # Adjust filename to put the suffix before '_mask'
    if '_mask' in base_name:
        new_base_name = base_name.replace('_mask', f"{suffix}_mask")
    else:
        new_base_name = base_name + suffix

    processed_image.save(os.path.join(destination_folder, f"{new_base_name}.png"))
    print(f"Processed and saved: {new_base_name}.png")

# Process each image in the source folder for images and masks
for filename in os.listdir(source_folder_images):
    if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.jpeg'):
        base_name = os.path.splitext(filename)[0]

        # Process image
        image_path = os.path.join(source_folder_images, filename)
        image = Image.open(image_path).convert('RGB')
        process_and_save(image, base_name, destination_folder_images, '_6')
        process_and_save(image, base_name, destination_folder_images, '_7')

# Process each mask in the source folder
for filename in os.listdir(source_folder_masks):
    if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.jpeg'):
        base_name = os.path.splitext(filename)[0]

        # Process mask (assuming masks have the same filename in the mask folder)
        mask_path = os.path.join(source_folder_masks, filename)
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert('RGB')
            process_and_save(mask, base_name, destination_folder_masks, '_6')
            process_and_save(mask, base_name, destination_folder_masks, '_7')

print("Processing complete.")


## Delete

In [ ]:
from PIL import Image
import os

# Define the path to the folder containing the images
folder_path = './dataset/segmentation/kvasir/medaugment/validation/'  # Replace with your actual folder path

# Target size for resizing
target_size = (224, 224)

# Get a list of all files in the folder
files_in_folder = os.listdir(folder_path)

# Loop through all files in the folder
for filename in files_in_folder:
    # Check if the file is an image (you can add more extensions if needed)
    if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')):
        # Construct the full path to the image file
        file_path = os.path.join(folder_path, filename)

        # Open the image using PIL
        with Image.open(file_path) as img:
            # Get the size of the image
            width, height = img.size

            # Print the filename and its size
            print(f"Image: {filename}, Size: {width}x{height}")

            # Check if the image size is not 224x224
            if (width, height) != target_size:
                print(f"Resizing image: {filename}")

                # Resize the image to 224x224
                resized_img = img.resize(target_size, Image.BILINEAR)

                # Save the resized image back to the same path (overwrite the original)
                resized_img.save(file_path)
                print(f"Image resized and saved: {filename}")

print("Image size check and resizing process complete.")


Streaming output truncated to the last 5000 lines.
Image resized and saved: cju5b9oyda4yr0850g9viziyv_3.jpg
Image: cju5b9oyda4yr0850g9viziyv_4.jpg, Size: 622x528
Resizing image: cju5b9oyda4yr0850g9viziyv_4.jpg
Image resized and saved: cju5b9oyda4yr0850g9viziyv_4.jpg
Image: cju5c5xc7algd0817pb1ej5yo_1.jpg, Size: 599x528
Resizing image: cju5c5xc7algd0817pb1ej5yo_1.jpg
Image resized and saved: cju5c5xc7algd0817pb1ej5yo_1.jpg
Image: cju5c5xc7algd0817pb1ej5yo_5.jpg, Size: 599x528
Resizing image: cju5c5xc7algd0817pb1ej5yo_5.jpg
Image resized and saved: cju5c5xc7algd0817pb1ej5yo_5.jpg
Image: cju5c5xc7algd0817pb1ej5yo_2.jpg, Size: 599x528
Resizing image: cju5c5xc7algd0817pb1ej5yo_2.jpg
Image resized and saved: cju5c5xc7algd0817pb1ej5yo_2.jpg
Image: cju5c5xc7algd0817pb1ej5yo_3.jpg, Size: 599x528
Resizing image: cju5c5xc7algd0817pb1ej5yo_3.jpg
Image resized and saved: cju5c5xc7algd0817pb1ej5yo_3.jpg
Image: cju5c5xc7algd0817pb1ej5yo_4.jpg, Size: 599x528
Resizing image: cju5c5xc7algd0817pb1ej5yo_4

In [ ]:
!python segmentation.py --model DeepLab --dataset kvasir --train_type segmentation --index 4 --seed 1 --decay True --bs 128 --num_class 1 --value 0.01 --epoch 40 --lr 0.002 --min_loss 10000

/usr/local/lib/python3.10/dist-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
43
Epoch: 0
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10
Epoch: 11
Epoch: 12
Epoch: 13
Epoch: 14
Epoch: 15
Epoch: 16
Epoch: 17
Epoch: 18


In [ ]:
Med_classification_VGG = [79.2, 88.6, 74.8, 76.6, 89.8, 75.1, 85.5, 91.4, 83.3, 82.0, 91.2, 82.4, 82.6, 94.5, 82.7, 81.7, 94.2, 81.3, 97.5, 97.5, 97.5, 97.5, 97.5, 97.5]
Med_classification_ConvNext = [80.5, 89.3, 82.3, 73.6, 86.9, 76.7, 85.5, 92.7, 88.2, 77.8, 88.9, 81.8, 82.5, 94.4, 82.2, 81.8, 94.2, 81.4, 95.9, 95.9, 95.9, 95.9, 95.9, 95.9]
Med_classification_ResNext = [77.3, 87.4, 78.7, 72.0, 85.0, 74.2, 81.2, 88.6, 77.5, 77.8, 89.0, 77.5, 84.6, 94.9, 84.3, 83.8, 94.9, 84.0, 95.0, 95.0, 95.0, 95.0, 95.0, 95.0]

Zoom_classification_VGG = [78.6, 88.9, 76.8, 70.3, 86.9, 72.5, 89.1, 93.1, 86.8, 88.1, 93.6, 87.4, 83.3, 94.8, 82.9, 82.4, 94.5, 81.9, 98.3, 98.3, 98.3, 98.3, 98.3, 98.3]
Zoom_classification_ConvNext = [81.2, 89.8, 84.1, 74.0, 87.2, 77.4, 84.8, 93.5, 91.8, 74.9, 87.4, 80.5, 81.2, 89.8, 84.1, 74.0, 87.2, 77.4, 96.7, 96.7, 96.7, 96.7, 96.7, 96.7]
Zoom_classification_ResNext = [77.3, 88.2, 82.1, 67.8, 84.1, 72.0, 83.3, 90.0, 81.4, 79.1, 89.3, 80.1, 82.4, 94.4, 81.9, 81.2, 94.1, 81.0, 98.4, 98.4, 98.4, 98.4, 98.4, 98.4]

def cal_avg(my_list):
  return round(sum(my_list) / len(my_list), 2)

print("----------------------------MedAugment---------------------------------------\n")
print(f"Avg MedAugment VGG: {cal_avg(Med_classification_VGG)}")
print(f"Avg MedAugment ConvNext: {cal_avg(Med_classification_ConvNext)}")
print(f"Avg MedAugment ResNext: {cal_avg(Med_classification_ResNext)}")

med_classification = Med_classification_VGG + Med_classification_ConvNext + Med_classification_ResNext
med_classification_avg = cal_avg(med_classification)

print("-----------------------------Zoom-in/out--------------------------------------\n")
print(f"Avg Zoom-in/out VGG: {cal_avg(Zoom_classification_VGG)}")
print(f"Avg Zoom-in/out ConvNext: {cal_avg(Zoom_classification_ConvNext)}")
print(f"Avg Zoom-in/out ResNext: {cal_avg(Zoom_classification_ResNext)}")

zoom_classification = Zoom_classification_VGG + Zoom_classification_ConvNext + Zoom_classification_ResNext
zoom_classification_avg = cal_avg(zoom_classification)

print("---------------------------Comparision----------------------------------------\n")
print(f"Overall comparision, zoom is {round(zoom_classification_avg - med_classification_avg , 2)}% better than MedAugment")
print(f"In VGG , zoom is {round(cal_avg(Zoom_classification_VGG) - cal_avg(Med_classification_VGG) , 2)}% better than MedAugment")
print(f"In ConvNext , zoom is {round(cal_avg(Zoom_classification_ConvNext) - cal_avg(Med_classification_ConvNext) , 2)}% worse than MedAugment")
print(f"In ResNext , zoom is {round(cal_avg(Zoom_classification_ResNext) - cal_avg(Med_classification_ResNext) , 2)}% better than MedAugment")

print(f"med: {med_classification_avg}")
print(f"zoom: {zoom_classification_avg}")

----------------------------MedAugment---------------------------------------

Avg MedAugment VGG: 87.58
Avg MedAugment ConvNext: 87.34
Avg MedAugment ResNext: 85.95
-----------------------------Zoom-in/out--------------------------------------

Avg Zoom-in/out VGG: 88.4
Avg Zoom-in/out ConvNext: 86.69
Avg Zoom-in/out ResNext: 86.67
---------------------------Comparision----------------------------------------

Overall comparision, zoom is 0.3% better than MedAugment
In VGG , zoom is 0.82% better than MedAugment
In ConvNext , zoom is -0.65% worse than MedAugment
In ResNext , zoom is 0.72% better than MedAugment
med: 86.95
zoom: 87.25


In [ ]:
len(Zoom_classification_ConvNext)

24

In [ ]:
med_seg_AllMetrics = [93.2, 87.6, 96.3, 61.5, 48.6, 92.9, 72.2, 61.1, 90.8, 92.7, 86.7, 95.9, 68.5, 56.9, 95.9, 69.2, 56.9, 90.8, 90.7, 83.3, 95.2, 70.6, 59.6, 96.7, 74.1, 63.1, 91.1]
med_seg_DC = [93.2, 61.5, 72.2, 92.7, 68.5, 69.2, 90.7, 70.6, 74.1]
med_seg_PA = [96.3, 92.9, 90.8, 95.9, 95.9, 90.8, 95.2, 96.7, 91.1]
med_seg_IOU = list(set(med_seg_AllMetrics) - set(med_seg_DC) - set(med_seg_PA))

zoom_seg_AllMetrics = [91.4, 84.6, 95.1, 59.2, 46.0, 92.3, 61.4, 49.1, 83.9, 92.2, 85.9, 95.6, 64.2, 50.9, 94.8, 63.8, 51.6, 89.4, 91.3, 84.1, 95.3, 62.8, 50.9, 94.7, 66.7, 54.5, 87.8]
zoom_seg_DC = [91.4, 59.2, 61.4, 92.2, 64.2, 63.8, 91.3, 62.8, 66.7]
zoom_seg_PA = [95.1, 92.3, 83.9, 95.6, 94.8, 89.4, 95.3, 94.7, 87.8]
zoom_seg_IOU = list(set(zoom_seg_AllMetrics) - set(zoom_seg_DC) - set(zoom_seg_PA))


print("------------------------------Random // Medaugment------------------------------------")
print(f"Avg of MedAugment of Random: {cal_avg(med_seg_AllMetrics)}")
print(f"Avg of Med DC of Random: {cal_avg(med_seg_DC)}%")
print(f"Avg of Med PA of Random: {cal_avg(med_seg_PA)}%")
print(f"Avg of Med IOU of Random: {cal_avg(med_seg_IOU)}%")

print("------------------------------Random // Zoom-in/out------------------------------------")
print(f"Avg of MedAugment of Random: {cal_avg(zoom_seg_AllMetrics)}")
print(f"Avg of Med DC of Random: {cal_avg(zoom_seg_DC)}%")
print(f"Avg of Med PA of Random: {cal_avg(zoom_seg_PA)}%")
print(f"Avg of Med IOU of Random: {cal_avg(zoom_seg_IOU)}%")



------------------------------Random // Medaugment------------------------------------
Avg of MedAugment of Random: 79.34
Avg of Med DC of Random: 76.97%
Avg of Med PA of Random: 93.96%
Avg of Med IOU of Random: 68.36%
------------------------------Random // Zoom-in/out------------------------------------
Avg of MedAugment of Random: 75.54
Avg of Med DC of Random: 72.56%
Avg of Med PA of Random: 92.1%
Avg of Med IOU of Random: 63.34%


In [41]:
UNet_med_seg = [93.2, 87.6, 96.3]
FPN_med_seg = [92.7, 86.7, 95.9]
DeepLab_med_seg = [90.7, 83.3, 95.2]

UNet_zoom_seg = [94.1, 89.1, 96.6]
FPN_zoom_seg = [93.9, 87.8, 96.6]
DeepLab_zoom_seg = [94.2, 89.2, 96.7]

print("------------------------------label-aware------------------------------------")
print(f"UNet: Our model performs by {round(cal_avg(UNet_zoom_seg) - cal_avg(UNet_med_seg), 2)}% worse than MedAugment")
print(f"FPN: Our model performs by {round(cal_avg(FPN_zoom_seg) - cal_avg( FPN_med_seg), 2)}% better than MedAugment")
print(f"DeepLab: Our model performs by {round(cal_avg(DeepLab_zoom_seg) - cal_avg(DeepLab_med_seg), 2)}% better than MedAugment")

------------------------------label-aware------------------------------------
UNet: Our model performs by 0.9% worse than MedAugment
FPN: Our model performs by 1.0% better than MedAugment
DeepLab: Our model performs by 3.64% better than MedAugment


In [ ]:
avg_zoom = cal_avg(UNet_zoom_seg)
avg_med = cal_avg(UNet_med_seg)

print(f"Our model performs by {round(avg_zoom - avg_med, 2)}% better than MedAugment")

Our model performs by -5.61% better than MedAugment
